In [0]:
%run ./utils

# Import

In [0]:
from datetime import date, timedelta
import holidays

# Load

In [0]:
dbutils.widgets.text("silver_table_name","tb_chamados_hora")
silver_table_name = dbutils.widgets.get("silver_table_name")

dbutils.widgets.text("gold_table_name", "dim_data")
gold_table_name = dbutils.widgets.get("gold_table_name")

In [0]:
df_src = load_df(silver_table_name)

# Lógica de Negócio

In [0]:
# Criar calendario do tempo de referencia
bounds = df_src.agg(
    F.min(F.least("hora_abertura_chamado", "hora_inicio_atendimento","hora_finalizacao_atendimento")).alias("dt_min"),
    F.max(F.greatest("hora_abertura_chamado", "hora_inicio_atendimento","hora_finalizacao_atendimento")).alias("dt_max"),
).first()

if bounds.dt_min is None:
    raise ValueError(f"{silver_table_name} não tem nenhum timestamp válido")

ano_ini,ano_fim = bounds.dt_min.year,bounds.dt_max.year
print(f"Calendário: {ano_ini} a {ano_fim}")

In [0]:
df_cal = spark.range(1).select(
    F.explode(F.sequence(
        F.lit(date(ano_ini, 1, 1)), F.lit(date(ano_fim, 12, 31)), F.expr("INTERVAL 1 DAY")
    )).alias("data")
)

In [0]:
anos = range(ano_ini, ano_fim + 1)

# Nacionais (nomes em português). Sem subdiv, a lib retorna só os feriados do país.
br = holidays.Brazil(years=anos, language="pt_BR")

rows = sorted(br.items())   # [(date, nome), ...]
df_fer = spark.createDataFrame(rows, "data date, nome_feriado string")

# Transformação


In [0]:
# TRANSFORMAÇÃO
nomes_dia = ["Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado"]

dim = (
    df_cal
    .join(df_fer, "data", "left")                                   
    .withColumn("data_sk", F.date_format("data", "yyyyMMdd").cast("bigint"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana_num", F.dayofweek("data"))              # 1 = domingo ... 7 = sábado
    .withColumn("dia_semana", F.element_at(F.array(*[F.lit(n) for n in nomes_dia]), F.col("dia_semana_num")))
    .withColumn("fim_de_semana", F.col("dia_semana_num").isin(1, 7))
    .withColumn("feriado", F.col("nome_feriado").isNotNull())
    .withColumn("dia_util", ~F.col("fim_de_semana") & ~F.col("feriado"))
    .select("data_sk", "data", "ano", "mes", "dia", "dia_semana", "dia_semana_num",
            "dia_util", "feriado", "nome_feriado", "fim_de_semana")
)

# Surrogate Keys

# Validações

In [0]:
# VALIDAÇÕES (falham de verdade; a Gold utils ainda não tem dq_check)
n_dias = (date(ano_fim, 12, 31) - date(ano_ini, 1, 1)).days + 1
n_rows = dim.count()
n_sk   = dim.select("data_sk").distinct().count()
expected_rows = (date(ano_fim, 12, 31) - date(ano_ini, 1, 1)).days + 1

assert n_rows == expected_rows
assert n_sk == n_rows,       "data_sk duplicada"
assert dim.filter(F.col("data_sk").isNull()).count() == 0, "data_sk nula"

# Salvamento

In [0]:
# SAVE
save_df(dim, gold_table_name)

spark.sql(f"COMMENT ON TABLE {gold_schema}.{gold_table_name} IS "
          "'Dimensão de data. Grain: 1 por dia. SK yyyyMMdd; -1 = não informado. Feriados nacionais apenas.'")

In [0]:
display(dim.limit(10))

In [0]:
# Cada papel da data vira uma FK; nulo cai no membro -1
F.coalesce(F.date_format("hora_abertura_chamado", "yyyyMMdd").cast("bigint"), F.lit(-1)).alias("data_abertura_sk")
F.coalesce(F.date_format("hora_inicio_atendimento", "yyyyMMdd").cast("bigint"), F.lit(-1)).alias("data_inicio_sk")
F.coalesce(F.date_format("hora_finalizacao_atendimento", "yyyyMMdd").cast("bigint"), F.lit(-1)).alias("data_finalizacao_sk")